
### import needed packages and read the combined dataset

In [ ]:
import pandas as pd

In [ ]:
# Define dataset paths
dataset_files = [
    'CEAS_08.csv',
    'Nazario.csv',
    'Nazario_2.csv',
    'Nazario_5.csv',
    'Nigerian_5.csv',
    'Nigerian_Fraud.csv',
    'SpamAssasin.csv',
    'TREC_07.csv'
]

# Load datasets into dictionary
datasets = {}

for file in dataset_files:
    try:
        name = file.replace('.csv', '')
        datasets[name] = pd.read_csv(f'datasets/raw/{file}')
        print(f"✓ Loaded {file}")
    except Exception as e:
        print(f"✗ Error loading {file}: {e}")

print(f"\n✓ Successfully loaded {len(datasets)} datasets")

### Check all dataset dimensions columns etc, so we know how to properly combine them

In [ ]:
dimensions_df = pd.DataFrame({
    'Dataset': list(datasets.keys()),
    'Rows': [df.shape[0] for df in datasets.values()],
    'Columns': [df.shape[1] for df in datasets.values()],
    'Memory (MB)': [df.memory_usage(deep=True).sum() / 1024**2 for df in datasets.values()]
})

display(dimensions_df)

print(f"\nTotal Rows: {dimensions_df['Rows'].sum():,}")
print(f"Total Columns: {dimensions_df['Columns'].sum()}")
print(f"Total Memory: {dimensions_df['Memory (MB)'].sum():.2f} MB")

In [ ]:
for name, df in datasets.items():
    print(f"\n{'='*70}")
    print(f"DATASET: {name}")
    print(f"{'='*70}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

    print("Columns:")
    for i, col in enumerate(df.columns, 1):
        dtype = df[col].dtype
        null_count = df[col].isnull().sum()
        null_pct = (null_count / len(df)) * 100
        print(f"  {i:2d}. {col:30s} | {str(dtype):10s} | Nulls: {null_count:5d} ({null_pct:5.2f}%)")

    print(f"\n{'─'*70}")

### Combine the datasets into a single dataset and load the knew unified dataset

In [ ]:
# Add source column to track which dataset each row came from
for name, df in datasets.items():
    df['source_dataset'] = name
    print(f"✓ Added source column to {name}")

# Combine all datasets
combined_df = pd.concat(datasets.values(), ignore_index=True)

print(f"\n{'='*60}")
print(f"Combined Dataset Shape: {combined_df.shape[0]:,} rows × {combined_df.shape[1]} columns")
print(f"{'='*60}\n")

# Display value counts by source
print("Records per source dataset:")
display(combined_df['source_dataset'].value_counts().sort_index())

# Export to CSV
output_file = 'datasets/processed/combined_dataset.csv'
combined_df.to_csv(output_file, index=False)
print(f"\n✓ Combined dataset exported to: {output_file}")

In [ ]:
# Load the unified dataset
unified_df = pd.read_csv('datasets/processed/combined_dataset.csv')
unified_df